### Q1 


In [1]:
%pip install pandas==2.3.2

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd

In [3]:
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

raw_df = pd.read_csv(data_path)

In [4]:
print(len(raw_df))

467985


In [5]:
print(raw_df.columns)
print(raw_df["model"].unique)

Index(['id', 'adv_source_id', 'source_id', 'model', 'decoding',
       'repetition_penalty', 'attack', 'domain', 'title', 'prompt',
       'generation'],
      dtype='object')
<bound method Series.unique of 0            human
1            human
2            human
3            human
4            human
            ...   
467980    mpt-chat
467981    mpt-chat
467982    mpt-chat
467983    mpt-chat
467984    mpt-chat
Name: model, Length: 467985, dtype: object>


In [6]:
print(raw_df["model"].unique())


['human' 'llama-chat' 'mpt' 'mpt-chat' 'gpt2' 'mistral' 'mistral-chat'
 'gpt3' 'cohere' 'chatgpt' 'gpt4' 'cohere-chat']


In [7]:

print(f"using model{raw_df["model"][1000]}")
print(raw_df["generation"][1000])

using modelllama-chat
In this paper, we present a novel method for packing regular tetrahedra in three-dimensional space, resulting in an icosahedral quasicrystal. By arranging the tetrahedra in a specific way, we create a structure that exhibits long-range order and uniform density, yet lacks the usual periodicity of crystalline materials. The resulting quasicrystal has a fascinating geometry, with each tetrahedron intersection forming a unique network of triangular faces.

We explore the properties of this quasicrystal using computational methods, including molecular dynamics simulations and mathematical modeling. Our results show that the icosahedral quasicrystal exhibits remarkable mechanical strength and stability, making it a promising candidate for applications in advanced materials science. Additionally, we observe intriguing optical properties, including Bragg reflections that are forbidden in traditional crystals due to the lack of translational symmetry.

The discovery of th

In [8]:
df = raw_df[raw_df["model"].isin(["human", "cohere"])]

In [9]:
df.groupby("model").size()

model
cohere    26742
human     13371
dtype: int64

In [10]:
df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

/tmp/ipykernel_2898/3636558662.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)


In [11]:
%pip install scikit-learn==1.7.1 scipy==1.13.1 numpy==1.26.4

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [12]:
from sklearn.model_selection import train_test_split

In [ ]:
df.columns

Index(['id', 'adv_source_id', 'source_id', 'model', 'decoding',
       'repetition_penalty', 'attack', 'domain', 'title', 'prompt',
       'generation', 'is_human'],
      dtype='object')

In [14]:
X = df["generation"]
y = df["is_human"]
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=.2, random_state=67)



### Q2

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

In [16]:
vectorizer = CountVectorizer(lowercase=True, max_features=500)

In [17]:
X_train_bow = vectorizer.fit_transform(X_train)
X_val_bow = vectorizer.transform(X_val)

In [18]:
print("Features:", vectorizer.get_feature_names_out()[:30])

print("\nTraining set:")
print(X_train_bow.toarray()[0])

Features: ['000' '10' '12' '15' '20' '30' 'able' 'about' 'action' 'add' 'added'
 'after' 'again' 'against' 'age' 'all' 'almost' 'along' 'also' 'always'
 'am' 'american' 'an' 'and' 'another' 'any' 'anything' 'approach' 'are'
 'around']

Training set:
[ 0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  1  0  0  0  1 13
  0  2  0  0  3  0  0  4  1  0  0  0  0  0  0  6  0  0  0  0  1  0  0  0
  0  0  0  0  0  1  0  0  0  0  0  0  0  1  0  0  0  3  0  5  0  0  3  0
  0  0  0  0  0  0  3  0  0  1  0  0  0  0  3  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  1  1  0  1  1  0  1  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0
  1  5  0  0  1  0  0  0  3  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0
  1  0  2  0  0  0  0  0  0  0  0  0  0  0  1  3  0  1  0  0  0  0  0  2
  0  1  0  0  0  0  0  0  0  2  0  0  8  0  0  0  7  0  0  0  0  0  1  4
  8  0  0  0  0  1  0  0  2  0  0  1  0  0  0  0  0  0  0  0  0  3  0  5
  0  2  1  0  0  0  

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(lowercase=True, max_features=500)

In [25]:
def do_vect(X_train: pd.array, X_val: pd.array, use_vect: str, max_features: int):
    if not use_vect in ("bow","tf-idf"):
        raise ValueError 
    elif use_vect == "bow":
        vectorizer = CountVectorizer(lowercase=True, max_features=max_features)
    elif use_vect == "tf-idf":
        vectorizer = TfidfVectorizer(lowercase=True, max_features=max_features)
    
    X_train = vectorizer.fit_transform(X_train)
    X_val = vectorizer.transform(X_val)
    return X_train, X_val

In [26]:
X_train_bow, X_val_bow = do_vect(X_train,X_val, "bow", 500)
X_train_tfidf, X_val_tfidf = do_vect(X_train,X_val, "tf-idf", 500)

In [27]:
from sklearn.linear_model import LogisticRegression

In [28]:
# clf is common naming convention
clf_1 = LogisticRegression(
    random_state=42,
    solver="liblinear",   # tested all solvers, and this one was fastest ... (should also be appropriate for our data)
    max_iter=1000,        # allow more iterations
    C=1.0,                # adjust if needed (smaller values = stronger regularization)
)

In [29]:
clf_1.fit(X_train_bow, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [30]:
from sklearn.metrics import classification_report

In [31]:
y_pred = clf_1.predict(X_val_bow)

In [34]:
report = classification_report(y_val, y_pred) 

In [33]:
print(report)

              precision    recall  f1-score   support

           0       0.81      0.90      0.85      5392
           1       0.73      0.56      0.63      2631

    accuracy                           0.79      8023
   macro avg       0.77      0.73      0.74      8023
weighted avg       0.78      0.79      0.78      8023



In [35]:
# clf is common naming convention
clf_2 = LogisticRegression(
    random_state=42,
    solver="liblinear",   # tested all solvers, and this one was fastest ... (should also be appropriate for our data)
    max_iter=1000,        # allow more iterations
    C=1.0,                # adjust if needed (smaller values = stronger regularization)
)
clf_2.fit(X_train_tfidf, y_train)
y_pred2 = clf_2.predict(X_val_tfidf)
report = classification_report(y_val, y_pred2)
print(report)

              precision    recall  f1-score   support

           0       0.80      0.90      0.85      5392
           1       0.74      0.55      0.63      2631

    accuracy                           0.79      8023
   macro avg       0.77      0.73      0.74      8023
weighted avg       0.78      0.79      0.78      8023



In [36]:
from sklearn.neural_network import MLPClassifier

In [ ]:
clf = MLPClassifier(random_state=1, max_iter=10).fit(X_train_tfidf, y_train)
clf.predict_proba(X_val_tfidf)
clf.predict(X_val_tfidf)
clf.score(X_val_tfidf, y_val)

### Q3

In [43]:
def split_from_raw(raw_df:pd.DataFrame,subset:str, split_size):
    if subset not in raw_df["model"].unique():
        raise ValueError
    else:
        df = raw_df[raw_df["model"].isin(["human", subset])]
    df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

    X = df["generation"]
    y = df["is_human"]
    X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=split_size, random_state=67)
    return X_train, X_val, y_train, y_val

def do_vect(X_train: pd.array, X_val: pd.array, use_vect: str, max_features: int):
    if not use_vect in ("bow","tf-idf"):
        raise ValueError 
    elif use_vect == "bow":
        vectorizer = CountVectorizer(lowercase=True, max_features=max_features)
    elif use_vect == "tf-idf":
        vectorizer = TfidfVectorizer(lowercase=True, max_features=max_features)
    
    X_train = vectorizer.fit_transform(X_train)
    X_val = vectorizer.transform(X_val)
    return X_train, X_val

def clasification(X_train,y_train):
    """
    X_train and y_train must be vectorized
    """
    clf = LogisticRegression(
    random_state=42,
    solver="liblinear",   # tested all solvers, and this one was fastest ... (should also be appropriate for our data)
    max_iter=1000,        # allow more iterations
    C=1.0,                # adjust if needed (smaller values = stronger regularization)
    )
    clf.fit(X_train, y_train)
    return clf

def evaluate(X_val,y_val,clf):
    y_pred = clf.predict(X_val)
    report = classification_report(y_val, y_pred)
    return y_pred, report


def pipe(raw_df,subset,split_size, use_vect: str, max_features: int):
    X_train, X_val, y_train, y_val = split_from_raw(raw_df,subset, split_size)
    X_train_vec, X_val_vec = do_vect(X_train, X_val, use_vect, max_features)
    clf = clasification(X_train_vec,y_train)
    y_pred, report = evaluate(X_val_vec,y_val,clf)
    return report


    

In [ ]:
models = raw_df["model"].unique()
vects = ("bow","tfidf")
max_feats = (500,1000,2000)
reports = pd.DataFrame(columns = ("report","vect","max_features"))
for vect in vects:
    for feats in max_feats:
        for model in models:
            if model == "human":
                pass
            else:
                reports["report"] = pipe(raw_df,model,.2,vect,feats)
                reports["vect"] = vect
                reports["max_features"] = feats


/tmp/ipykernel_2898/2226588024.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)
/tmp/ipykernel_2898/2226588024.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)
/tmp/ipykernel_2898/2226588024.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in

0.8092982674809921